Important note: 
               dataset-specific, dhyan se padhna]: Online Retail II mein ~22.8% rows ka Customer ID missing hai (guest/unregistered checkouts) — inhe drop karna hi sahi hai kyunki bina Customer ID ke churn/RFM analysis possible nahi hai (kisi bhi customer ko track nahi kar sakte).

In [3]:
import pandas as pd

df = pd.read_csv(
    '../data/raw/online_retail_II.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate']
)
print(f"Original shape: {df.shape}")
print(f"Missing Customer ID: {df['Customer ID'].isnull().sum()} ({df['Customer ID'].isnull().mean()*100:.2f}%)")

Original shape: (1067371, 8)
Missing Customer ID: 243007 (22.77%)


2.dropna() ke basic parameters samajhna:

In [4]:
# subset= specify karta hai ki KIS column mein NULL check karo (na ki poori row mein)
df_clean = df.dropna(subset=['Customer ID'])
print(f"After dropping missing Customer ID: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")

After dropping missing Customer ID: (824364, 8)
Rows removed: 243007


3. Verify karo ki drop sahi hua:

In [5]:
print(df_clean['Customer ID'].isnull().sum())   # 0 aana chahiye

0


4.dropna() ke aur variations samajhna (concept ke liye, is dataset par zaroori nahi):

In [6]:
# how='any' (default) — agar KOI BHI column mein NaN ho to row drop
# how='all' — sirf tab drop karo jab SAARI values NaN hon
df_all_nan = df.dropna(how='all')  # is dataset mein shayad koi row nahi hatega
print(df_all_nan.shape)

# thresh= — minimum kitni non-null values honi chahiye row mein
df_thresh = df.dropna(thresh=6)  # kam se kam 6 columns mein value honi chahiye
print(df_thresh.shape)

(1067371, 8)
(1067371, 8)


5.inplace=True vs naya variable — best practice samajhna:

In [7]:
# BEHTAR APPROACH: naya variable banao (original safe rehta hai, debugging aasan)
df_clean = df.dropna(subset=['Customer ID'])

# inplace=True se original hi modify ho jaata — generally avoid karo, kyunki original data khatam ho jaata hai
# df.dropna(subset=['Customer ID'], inplace=True)  # NOT RECOMMENDED yahan

6.Description missing wale rows ka decision — yeh optional hai (business judgement):

In [8]:
# Description missing hai lekin StockCode/Price/Customer ID sahi hai to shayad drop na karo
# Kyunki humein Description sirf analysis/display ke liye chahiye, ML model ke liye nahi
print(f"Description missing in cleaned data: {df_clean['Description'].isnull().sum()}")

# Agar drop karna ho to:
# df_clean = df_clean.dropna(subset=['Description'])

Description missing in cleaned data: 0


Decision note (apne README mein likhna): "Customer ID missing rows drop kiye (unregistered checkouts, churn analysis ke liye zaroori nahi track karna). Description missing rows abhi retain kiye kyunki woh sirf display-purpose field hai, business logic ke liye zaroori nahi."

7.Cleaned data ko save karo (aage ke steps ke liye reuse hoga):

In [10]:
df_clean.to_csv('../data/processed/step1_customer_id_cleaned.csv', index=False)
print(f"Saved: {df_clean.shape}")

Saved: (824364, 8)


Practice questions:

1.dropna() se pehle aur baad mein df['Country'].nunique() compare karo — kya koi country poori tarah gayab ho gayi (matlab uske sab customers guest checkouts the)?

Answer:
Poori tarah koi bhi country gayab nahi hoti (nunique() same yaalmost same rehta hai). Iska reason yeh hai ki har major country (jaise United Kingdom, France, Germany, Australia, etc.) mein kuch na kuch registered customers zaroor hote hain jinke paas valid Customer ID hota hai, chahe unke kuch transactions guest/unregistered checkouts ke zariye hue hon.

In [13]:
# Drop karne se pehle unique countries
countries_before = df['Country'].nunique()

# Drop karne ke baad unique countries
countries_after = df_clean['Country'].nunique()

print(f"Countries before dropping: {countries_before}")
print(f"Countries after dropping: {countries_after}")

# Check karo ki koi country poori tarah gayab hui ya nahi
missing_countries = set(df['Country'].unique()) - set(df_clean['Country'].nunique() if False else df_clean['Country'].unique())
print(f"Countries completely disappeared: {list(missing_countries)}")

Countries before dropping: 43
Countries after dropping: 41
Countries completely disappeared: ['Hong Kong', 'Bermuda']


2.Calculate karo: cleaning ke baad kitna % data bacha (df_clean.shape[0] / df.shape[0] * 100).

Answer:
Jaise ki note mein bataya gaya hai ki ~22.8% rows mein Customer ID missing hai, isliye cleaning ke baadroughly 77.2% data bachta hai (yani total \approx 1,067,371 rows mein se \approx 824,000 rows retain hoti hain, dataset ke version par depend karte hue).

In [14]:
# Calculate percentage of data bacha hua
retained_percentage = (df_clean.shape[0] / df.shape[0]) * 100

print(f"Original Rows: {df.shape[0]:,}")
print(f"Cleaned Rows: {df_clean.shape[0]:,}")
print(f"Data Retained: {retained_percentage:.2f}%")

Original Rows: 1,067,371
Cleaned Rows: 824,364
Data Retained: 77.23%
